In [ ]:
import json
import os
import base64
import pickle
import tempfile
from datetime import datetime
from typing import List, Dict, Tuple

import pandas as pd
from snowflake.snowpark.functions import col
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

# --------------------------------------------------
# Snowflake session
# --------------------------------------------------
session = get_active_session()

# --------------------------------------------------
# Constants
# --------------------------------------------------
COEFF_TABLE = "PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS"
REGISTRY_DB = "ORANGE_ZONE_SBX_TA"
REGISTRY_SCHEMA = "REGISTRY"

# --------------------------------------------------
# Custom Model Wrapper
# --------------------------------------------------
class InterceptCustomModel(custom_model.CustomModel):
    def __init__(self, model):
        super().__init__(context=None)
        self.model = model

    @custom_model.inference_api
    def predict(self, input: pd.DataFrame) -> pd.DataFrame:
        preds = self.model.predict(
            data=input,
            coeffs=self.model.model_coefficients_final
        )
        return pd.DataFrame({"prediction": preds})

# --------------------------------------------------
# Registry
# --------------------------------------------------
registry = Registry(
    session=session,
    database_name=REGISTRY_DB,
    schema_name=REGISTRY_SCHEMA
)

# --------------------------------------------------
# Data Fetch Helpers
# --------------------------------------------------
def fetch_intercept_rows():
    return (
        session.table(COEFF_TABLE)
        .filter(col("PARAMETERS") == "Intercept")
        .select(
            "REGIONNAME", "F_CODE", "MODEL_BYTES",
            "RSQ", "WMAPE", "MAPE", "RMSE", "BIAS", "TRACKING_SIGNAL",
            "ALPHA", "LAMBDA",
            "TRAIN_START_DATE", "TRAIN_END_DATE",
            "LOAD_TS"
        )
        .collect()
    )

def fetch_features(region: str, f_code: str) -> List[str]:
    rows = (
        session.table(COEFF_TABLE)
        .filter(
            (col("REGIONNAME") == region) &
            (col("F_CODE") == f_code)
        )
        .select("PARAMETERS")
        .collect()
    )
    return [r["PARAMETERS"] for r in rows]

# --------------------------------------------------
# Versioning (Semantic)
# --------------------------------------------------
def compute_semantic_version(
    region_fcode: str,
    features: List[str],
    train_start: str,
    train_end: str,
) -> str:
    metadata_file = f"model_metadata_{region_fcode}.json"

    if not os.path.exists(metadata_file):
        return "v_0_0"

    with open(metadata_file, "r") as f:
        last = json.load(f)

    last_features = last["features"]
    last_period = last["train_period"]
    major, minor = map(int, last["version"].replace("v_", "").split("_"))

    if features != last_features:
        return f"v_{major + 1}_0"

    if train_start != last_period["start"] or train_end != last_period["end"]:
        return f"v_{major}_{minor + 1}"

    return f"v_{major}_{minor}"

# --------------------------------------------------
# Registry Version (Sequential)
# --------------------------------------------------
def get_next_registry_version(registry: Registry, model_name: str) -> str:
    try:
        versions = registry.list_model_versions(model_name)
    except Exception:
        return "v0"

    nums = [
        int(v.version_name[1:])
        for v in versions
        if v.version_name.startswith("v") and v.version_name[1:].isdigit()
    ]

    return f"v{max(nums) + 1}" if nums else "v0"

# --------------------------------------------------
# Metadata Builder
# --------------------------------------------------
def build_model_metadata(
    *,
    region: str,
    f_code: str,
    version: str,
    features: List[str],
    row
) -> Dict:
    return {
        "name": f"{region}_{f_code}",
        "model_type": "ElasticNet",
        "description": f"ElasticNet demand model for {region}_{f_code}",
        "version": version,
        "metrics": {
            "rsq": float(row["RSQ"]),
            "wmape": float(row["WMAPE"]),
            "mape": float(row["MAPE"]),
            "rmse": float(row["RMSE"]),
            "bias": float(row["BIAS"]),
            "tracking_signal": float(row["TRACKING_SIGNAL"]),
        },
        "hyperparameters": {
            "alpha": float(row["ALPHA"]),
            "lambda": float(row["LAMBDA"]),
        },
        "train_period": {
            "start": str(row["TRAIN_START_DATE"]),
            "end": str(row["TRAIN_END_DATE"]),
        },
        "features": features,
        "training_date": str(row["LOAD_TS"]),
        "tags": {
            "region": region,
            "f_code": f_code,
            "algorithm": "ElasticNet"
        }
    }

# --------------------------------------------------
# Artifact Writer
# --------------------------------------------------
def write_metadata_artifact(region_fcode_safe: str, metadata: Dict) -> str:
    path = os.path.join(
        tempfile.gettempdir(),
        f"model_metadata_{region_fcode_safe}.json"
    )
    with open(path, "w") as f:
        json.dump(metadata, f, indent=2)
    return path

# --------------------------------------------------
# Main Loop
# --------------------------------------------------
for row in fetch_intercept_rows():
    region = row["REGIONNAME"]
    f_code = row["F_CODE"]
    region_fcode = f"{region}_{f_code}"
    region_fcode_safe = region_fcode.replace("-", "_")

    print(f"Processing {region_fcode}")

    # Deserialize model
    model = pickle.loads(base64.b64decode(row["MODEL_BYTES"]))
    custom_model_instance = InterceptCustomModel(model)

    # Features
    features = fetch_features(region, f_code)

    # Semantic version
    semantic_version = compute_semantic_version(
        region_fcode,
        features,
        str(row["TRAIN_START_DATE"]),
        str(row["TRAIN_END_DATE"])
    )

    # Metadata
    metadata = build_model_metadata(
        region=region,
        f_code=f_code,
        version=semantic_version,
        features=features,
        row=row
    )

    artifact_path = write_metadata_artifact(region_fcode_safe, metadata)

    # Registry version
    model_name = f"intercept_model_{region_fcode_safe}"
    registry_version = get_next_registry_version(registry, model_name)

    mv = registry.log_model(
        model=custom_model_instance,
        model_name=model_name,
        version_name=registry_version,
        conda_dependencies=["scikit-learn", "pandas"],
        options={"relax_version": False},
        user_files={"metadata": [artifact_path]},
        comment=f"Custom Intercept Model for {region_fcode}",
        sample_input_data=pd.DataFrame(model.X)
    )

    mv.set_alias("latest")

    print(
        f"Logged {model_name} | "
        f"semantic={semantic_version}, registry={registry_version}"
    )
